In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

In [2]:
#Dataset
torch.random.manual_seed(42)
unlabeled = 20
num_classes = 2
label_budget = 4

#data = torch.empty(unlabeled, num_classes).uniform_(-1, 1)
data = torch.cartesian_prod(
    torch.linspace(-1, 1, 5),  # 5x4=20
    torch.linspace(-1, 1, 4)
)
labels = (data[:, 1] >= data[:, 0]).float().unsqueeze(1)

all_subsets = list(combinations(range(unlabeled), label_budget))
x_all = torch.stack([data[list(subset)] for subset in all_subsets])
y_all = torch.stack([labels[list(subset)] for subset in all_subsets])

In [3]:
# x @ W.T + b
N = len(all_subsets) #number of subsets of size 4 from 20 samples = C(20, 4) = 4845
print(N)
W = torch.randn(N, 1, 2, requires_grad=True)
b = torch.randn(N, 1, requires_grad=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD([W, b], lr=0.1)


4845


In [4]:
# Training Loop
losses = []
for epoch in range(100):
    optimizer.zero_grad()
    logits = x_all @ W.transpose(-1, -2) + b.unsqueeze(1)
    loss = criterion(logits, y_all)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f'Final Loss: {loss.item():.4f}')

Final Loss: 0.8998


In [5]:
#test dataset
test_data = torch.cartesian_prod(torch.linspace(-1, 1, 20), torch.linspace(-1, 1, 20))
test_labels = (test_data[:, 1] >= test_data[:, 0]).float().unsqueeze(1)
test_dataset = TensorDataset(test_data, test_labels)
test_loader = DataLoader(test_dataset, batch_size=400, shuffle=False)

In [6]:
# Evaluate test dataset
with torch.no_grad():
    for x, y in test_loader:
        logits = x.unsqueeze(0) @ W.transpose(-1, -2) + b.unsqueeze(1)
        preds = (logits >= 0).float()
        accuracies = (preds == y.unsqueeze(0)).float().mean(dim=[1, 2])

#print(accuracies)
print(f'Random Sampling baseline: {accuracies.mean():.4f}')
#print(f'best subset accuracy: {accuracies.max():.4f}')
#print(f'worst subset accuracy: {accuracies.min():.4f}')
print(x.shape)

Random Sampling baseline: 0.5048
torch.Size([400, 2])


In [7]:
# Visualization of NN accuracy distribution across all subsets
plt.figure(figsize=(18, 12))

# get hist return values: n (count), bins (edges), patches (bar objects)
n, bins, patches = plt.hist(accuracies.numpy(), bins=50, range=(0, 1), edgecolor='black', color='steelblue', alpha=0.5)

# get number of subests for each accuracy bin
for i in range(len(patches)):
    count = n[i]
    if count > 0: # only show number for bars with count > 0
        # get bar center x and height y
        x = patches[i].get_x() + patches[i].get_width() / 2
        y = patches[i].get_height()
        
        # show count above the bar, with a small offset (max(n)*0.01) to avoid overlap with the bar
        plt.text(x, y + (max(n)*0.01), str(int(count)), 
                 ha='center', va='bottom', fontsize=12, rotation=60)
        
plt.axvline(np.mean(accuracies.numpy()), color='blue', linestyle='--', label=f'Mean: {np.mean(accuracies.numpy()):.4f}')
plt.xlabel('Accuracy')
plt.ylabel('Number of Subsets')
plt.title(f'Distribution of Test Accuracy across all {len(all_subsets)} (n={unlabeled}, k={label_budget})Subsets w/ Counts')
plt.legend(fontsize=24)
plt.tight_layout()
plt.savefig(f'Distribution {len(all_subsets)}, n={unlabeled}, k={label_budget} with Counts equal.png')


In [10]:
# Random Forest on toy dataset test for all subsets
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

accuracies_rf = []
count = 0
print(f"--- Random Forest Training and Evaluating for All {len(all_subsets)} Subsets ---")
for idx in range(len(all_subsets)):
    subset_idx = all_subsets[idx]
    x_train_rf = data[list(subset_idx)].cpu().numpy()
    y_train_rf = labels[list(subset_idx)].flatten().cpu().numpy()
    
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
    rf_model.fit(x_train_rf, y_train_rf)
    
    x_test_np = test_data.cpu().numpy()
    y_test_np = test_labels.flatten().cpu().numpy()

    rf_preds = rf_model.predict(x_test_np)
    rf_acc = accuracy_score(y_test_np, rf_preds)
    accuracies_rf.append(rf_acc)
    count += 1
    if count % 1000 == 0:
        print(f" {count} / {len(all_subsets)} completed")


--- Random Forest Training and Evaluating for All 4845 Subsets ---
 1000 / 4845 completed
 2000 / 4845 completed
 3000 / 4845 completed
 4000 / 4845 completed


In [11]:
fig, ax = plt.subplots(figsize=(10, 5))
common_range = (0, 1)
common_bins = 50
ax.hist(accuracies.numpy(), bins=common_bins, range=common_range, alpha=0.5, edgecolor='black', label='Neural Network')
ax.hist(accuracies_rf,      bins=common_bins, range=common_range, alpha=0.5, edgecolor='black', label='Random Forest')
ax.set_xlabel('Accuracy')
ax.set_ylabel('Number of Subsets')
ax.set_title(f'NN vs Random Forest: Distribution of Test Accuracy {len(all_subsets)} (n={unlabeled}, k={label_budget})')
ax.legend()

table_data = [
    [f'{accuracies.max():.4f}',  f'{np.max(accuracies_rf):.4f}'],
    [f'{accuracies.mean():.4f}', f'{np.mean(accuracies_rf):.4f}'],
    [f'{accuracies.min():.4f}',  f'{np.min(accuracies_rf):.4f}'],
]

table = ax.table(
    cellText=table_data,
    rowLabels=['Max', 'Mean', 'Min'],
    colLabels=['NN', 'RF'],
    loc='top left',        
    cellLoc='center',        
    bbox=[0.06, 0.78, 0.2, 0.2]
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(0.3, 0.6) 

plt.tight_layout()
plt.savefig(f'NN_vs_RF_distribution(n={unlabeled}, k={label_budget}, {len(all_subsets)}) equal.png')